# ETL Bronze - ECMWF PF (TIGGE ensemble via cdsapi)

Carga los JSON diarios de pf (50 miembros del ensemble, todo el bounding box ya recortado server-side por la API) en una tabla Bronze idempotente.

In [ ]:
import re
from datetime import date, timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType
from pyspark.sql.window import Window

BRONZE_TABLE = 'weather.bronze.ecmwf_forecast_pf'
RAW_PATH = '/Volumes/weather/raw/ecmwf_volume/pf_tigge/json/'

schema = StructType([
    StructField('run_date', StringType(), True),
    StructField('run_time', StringType(), True),
    StructField('step_hours', IntegerType(), True),
    StructField('valid_datetime', StringType(), True),
    StructField('valid_date', StringType(), True),
    StructField('latitude', DoubleType(), True),
    StructField('longitude', DoubleType(), True),
    StructField('number', IntegerType(), True),
    StructField('tp_mm', DoubleType(), True),
    StructField('tipo', StringType(), True),
    StructField('source_api', StringType(), True),
    StructField('extracted_at', StringType(), True),
])

In [ ]:
# Widgets de backfill acotado. Vacios (el default, y lo que usa el job
# ECMWF_Forecast_Historic_Backfill) => comportamiento historico: se lee el directorio entero
# del volumen. Con valores YYYY-MM-DD se leen SOLO los archivos de ese rango de run_date.
#
# `lookback_days` es la via del job diario: deriva el rango de las ultimas N fechas respecto de
# hoy. Sin el, la corrida diaria tendria que barrer los ~3.200 archivos de ~286 MB (~900 GB) del
# historico completo de pf para incorporar el unico dia nuevo que llego. Los widgets explicitos
# de rango ganan si estan puestos (un backfill a mano manda sobre el default del job).
try:
    dbutils.widgets.text('range_start', '')
    dbutils.widgets.text('range_end', '')
    dbutils.widgets.text('lookback_days', '')
    range_start = dbutils.widgets.get('range_start').strip() or None
    range_end = dbutils.widgets.get('range_end').strip() or None
    lookback_days = dbutils.widgets.get('lookback_days').strip() or None
except Exception:
    range_start = None
    range_end = None
    lookback_days = None

if lookback_days and not (range_start or range_end):
    hoy = date.today()
    range_start = (hoy - timedelta(days=int(lookback_days))).isoformat()
    range_end = hoy.isoformat()
    print(f'lookback_days={lookback_days} => rango derivado {range_start} .. {range_end}')

print(f'range_start={range_start}, range_end={range_end}')

In [ ]:
try:
    raw_files = [item.path for item in dbutils.fs.ls(RAW_PATH) if item.path.endswith('.json')]
except Exception:
    raw_files = []

if not raw_files:
    dbutils.notebook.exit(f'No hay archivos JSON en {RAW_PATH}')

if range_start or range_end:
    # Backfill acotado. El nombre del archivo codifica el dia (ECMWF_PF_YYYY_MM_DD_t00.json),
    # asi que se arma la lista explicita de paths del rango y se la pasa a spark.read.json()
    # en vez de leer el directorio entero: con 3.100+ archivos de ~286 MB (~890 GB) leer todo
    # para quedarse con un chunk hace que el parseo del JSON y la ventana de deduplicacion
    # barran los ~2.690 millones de filas del historico completo en cada corrida.
    # El resultado por dia es identico porque hay un unico archivo por (run_date, run_time) y
    # la ventana de dedup ya particiona por dia: acotar la lectura no puede sacar de la
    # particion de un dia ninguna fila que estuviese en la lectura completa.
    start = date.fromisoformat(range_start) if range_start else None
    end = date.fromisoformat(range_end) if range_end else None
    fname_re = re.compile(r'ECMWF_PF_(\d{4})_(\d{2})_(\d{2})_t\d{2}\.json$')
    read_paths, unparsed = [], []
    for path in raw_files:
        m = fname_re.search(path)
        if not m:
            unparsed.append(path)
            continue
        file_date = date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        if (start is None or file_date >= start) and (end is None or file_date <= end):
            read_paths.append(path)
    read_paths.sort()
    if unparsed:
        print(f'AVISO: {len(unparsed)} archivos con nombre no parseable, excluidos del rango: {unparsed[:5]}')
    print(f'{len(read_paths)} archivos en el rango [{range_start}, {range_end}] de {len(raw_files)} en el volumen')
    if not read_paths:
        dbutils.notebook.exit(f'No hay archivos JSON en el rango [{range_start}, {range_end}]')
else:
    read_paths = RAW_PATH

raw_df = spark.read.schema(schema).option('multiLine', True).json(read_paths)

bronze_df = (
    raw_df
    .withColumn('run_date', F.to_date('run_date'))
    .withColumn('valid_date', F.to_date('valid_date'))
    .withColumn('valid_datetime', F.to_timestamp('valid_datetime'))
    .withColumn('source_file', F.col('_metadata.file_path'))
    .withColumn('extracted_at_ts', F.to_timestamp('extracted_at'))
    .withColumn('ingestion_date', F.current_date())
    .withColumn('loaded_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .filter(F.col('run_date').isNotNull())
)

window = Window.partitionBy('run_date', 'run_time', 'step_hours', 'latitude', 'longitude', 'number').orderBy(F.col('extracted_at_ts').desc_nulls_last())
bronze_df = (
    bronze_df.withColumn('row_number', F.row_number().over(window))
    .filter(F.col('row_number') == 1)
    .drop('row_number')
    .select(
        'run_date', 'run_time', 'step_hours', 'valid_date', 'valid_datetime', 'latitude', 'longitude', 'number',
        'tp_mm', 'tipo', 'source_api', 'source_file', F.col('extracted_at_ts').alias('extracted_at'),
        'ingestion_date', 'loaded_at', 'updated_at',
    )
)

if bronze_df.limit(1).count() == 0:
    dbutils.notebook.exit('No valid rows to merge')

merge_condition = 't.run_date = s.run_date AND t.run_time = s.run_time AND t.step_hours = s.step_hours AND t.latitude = s.latitude AND t.longitude = s.longitude AND t.number <=> s.number'
if range_start:
    # Acota el lado target por run_date para que Delta descarte archivos por estadisticas.
    # No cambia el resultado: el source solo trae filas del rango, asi que una fila del
    # target fuera del rango nunca podria matchear la condicion de join.
    merge_condition += f" AND t.run_date >= DATE'{range_start}'"
if range_end:
    merge_condition += f" AND t.run_date <= DATE'{range_end}'"

DeltaTable.forName(spark, BRONZE_TABLE).alias('t').merge(
    bronze_df.alias('s'),
    merge_condition,
).whenNotMatchedInsertAll().execute()

spark.table(BRONZE_TABLE).agg(F.min('run_date').alias('inicio'), F.max('run_date').alias('fin'), F.count('*').alias('rows')).show()